# Day 22 Tutorial：识别训练内预测与分组泄漏

> **课程附带的确定性玩具演示。** 这里的分组合成数据和数值不是 ESOL，更不是粘合剂实验结果。

## Goal

用同一组带 group 的玩具回归数据对比“训练内预测训练二层”和“GroupKFold OOF 预测训练二层”的信息路径，并验证外部组与每个内部折的组都互不重叠。


## Setup

每组有四个相近样本。先按 group 切出独立外部 valid，再只在外部训练组中生成 OOF。此设计用于显示信息边界；一次数值方向不是定理。


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

SEED = 42
rng = np.random.default_rng(SEED)
n_groups = 30
rows_per_group = 4
groups = np.repeat(np.arange(n_groups), rows_per_group)
group_centers = rng.normal(size=(n_groups, 8))
X = (
    group_centers[groups]
    + rng.normal(scale=0.15, size=(len(groups), 8))
)
coefficients = np.array([8, -6, 4, 0, 3, -2, 0, 5], dtype=float)
group_effects = rng.normal(scale=12.0, size=n_groups)
y = (
    X @ coefficients
    + group_effects[groups]
    + rng.normal(scale=2.0, size=len(groups))
)

outer_split = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=SEED
)
train_idx, valid_idx = next(
    outer_split.split(X, y, groups=groups)
)
X_train, X_valid = X[train_idx], X[valid_idx]
y_train, y_valid = y[train_idx], y[valid_idx]
groups_train, groups_valid = groups[train_idx], groups[valid_idx]

base_models = [
    ("tree", DecisionTreeRegressor(random_state=SEED)),
    ("ridge", make_pipeline(StandardScaler(), Ridge(alpha=1.0))),
]
assert set(groups_train).isdisjoint(set(groups_valid))
print("Train / valid:", X_train.shape, X_valid.shape)
print(
    "Train / valid groups:",
    len(np.unique(groups_train)),
    len(np.unique(groups_valid)),
)


Train / valid: (88, 8) (32, 8)
Train / valid groups: 22 8


## Steps

### 1. 构造故意错误的训练内二层特征

基础模型先看过全部训练样本，再预测同一批样本。这段代码只用于暴露风险，不能作为正确 stacking 模板。


In [2]:
fitted_leaky = []
leaky_columns = []
for _, estimator in base_models:
    fitted = clone(estimator).fit(X_train, y_train)
    fitted_leaky.append(fitted)
    leaky_columns.append(fitted.predict(X_train))
leaky_meta_X = np.column_stack(leaky_columns)
leaky_meta = Ridge(alpha=1.0).fit(leaky_meta_X, y_train)
print("Leaky meta input shape:", leaky_meta_X.shape)


Leaky meta input shape: (88, 2)


### 2. 用 GroupKFold 逐折生成 OOF 二层特征

每个位置只由没见过该位置标签、也没见过该位置所属 group 的折模型写入。


In [3]:
inner_splits = list(
    GroupKFold(n_splits=5).split(
        X_train, y_train, groups=groups_train
    )
)
oof_meta_X = np.empty((len(y_train), len(base_models)))
coverage = np.zeros(len(y_train), dtype=int)

for fit_idx, hold_idx in inner_splits:
    assert set(groups_train[fit_idx]).isdisjoint(
        set(groups_train[hold_idx])
    )
    coverage[hold_idx] += 1
    for column, (_, estimator) in enumerate(base_models):
        fold_model = clone(estimator)
        fold_model.fit(X_train[fit_idx], y_train[fit_idx])
        oof_meta_X[hold_idx, column] = fold_model.predict(
            X_train[hold_idx]
        )

oof_meta = Ridge(alpha=1.0).fit(oof_meta_X, y_train)
print("OOF meta input shape:", oof_meta_X.shape)
print("Coverage values:", np.unique(coverage, return_counts=True))


OOF meta input shape: (88, 2)
Coverage values: (array([1]), array([88]))


### 3. 对同一组独立外部 valid 评价

两个二层模型都使用基础模型在完整训练集上重拟合后的外部预测。训练内预测通常会让二层训练误差看起来过好，但外部 RMSE 的方向并不保证。


In [4]:
final_base_models = [
    clone(estimator).fit(X_train, y_train)
    for _, estimator in base_models
]
valid_meta_X = np.column_stack([
    model.predict(X_valid) for model in final_base_models
])
comparison = pd.DataFrame([
    {
        "meta_training_input": "leaky in-sample predictions",
        "meta_train_rmse": root_mean_squared_error(
            y_train, leaky_meta.predict(leaky_meta_X)
        ),
        "external_group_valid_rmse": root_mean_squared_error(
            y_valid, leaky_meta.predict(valid_meta_X)
        ),
    },
    {
        "meta_training_input": "group-out-of-fold predictions",
        "meta_train_rmse": root_mean_squared_error(
            y_train, oof_meta.predict(oof_meta_X)
        ),
        "external_group_valid_rmse": root_mean_squared_error(
            y_valid, oof_meta.predict(valid_meta_X)
        ),
    },
])
display(comparison.round(3))


,meta_training_input,meta_train_rmse,external_group_valid_rmse
0,leaky in-sample predictions,0.001,20.793
1,group-out-of-fold predictions,13.993,18.457


## Checks

覆盖与组互斥断言共同检查索引边界；还要人工确认每折预处理都在 Pipeline 内，外部验证标签只用于最终指标。


In [5]:
assert set(groups_train).isdisjoint(set(groups_valid))
assert np.all(coverage == 1)
for fit_idx, hold_idx in inner_splits:
    assert set(groups_train[fit_idx]).isdisjoint(
        set(groups_train[hold_idx])
    )
assert oof_meta_X.shape == leaky_meta_X.shape == (len(y_train), 2)
assert np.isfinite(oof_meta_X).all()
assert np.isfinite(
    comparison[[
        "meta_train_rmse",
        "external_group_valid_rmse",
    ]]
).all().all()
print("Checks passed: outer and inner group boundaries are disjoint.")


Checks passed: outer and inner group boundaries are disjoint.


## Next Steps

Day 23 从 ESOL SMILES 构造 Bemis–Murcko scaffold groups，把 `GroupKFold` 生成的 split 列表传给 `StackingRegressor`。真实粘合剂必须使用化学组确认的配方、批次或实验组字段。
